# 02. 収集データのレビュー

`01_collect_data.ipynb` で集めたデータ(各エピソードの `video.mp4` + `actions.csv`)を可視化して確認します。
学習 (`03_train_bc.ipynb`) に進む前に、意図した通りのデータが集まっているかをここで確認してください。
このノートブックは読み取り専用で、データ収集や学習には影響しません。

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
EPISODES_DIR = DATA_DIR / "episodes"

manifest = pd.read_csv(MANIFEST_PATH)
manifest

## 要約統計

In [ ]:
print(f"エピソード数: {len(manifest)}")
print(f"クリア率 (flag_get): {manifest['flag_get'].mean():.1%}")
manifest[["num_frames", "score", "x_pos_max", "coins", "time_remaining"]].describe()

## アクション分布 (全エピソード合計)

学習時のクラス不均衡を事前に把握するために、7種類のアクションそれぞれが何フレーム記録されたかを集計する。

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

ACTION_NAMES = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]

action_counts = Counter()
for ep_dir in sorted(EPISODES_DIR.glob("ep_*")):
    actions_csv = ep_dir / "actions.csv"
    if not actions_csv.exists():
        continue
    action_counts.update(pd.read_csv(actions_csv)["action"].tolist())

counts = [action_counts.get(i, 0) for i in range(len(SIMPLE_MOVEMENT))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ACTION_NAMES, counts)
ax.set_ylabel("フレーム数")
ax.set_title("アクションの出現回数")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## エピソードのフレームサンプル表示

`episode_id` を変えて、別のエピソードを確認できる。

In [ ]:
import cv2
import numpy as np

episode_id = manifest["episode_id"].iloc[0]  # 確認したいエピソードIDに変更してください
ep_dir = EPISODES_DIR / episode_id
actions_df = pd.read_csv(ep_dir / "actions.csv")

cap = cv2.VideoCapture(str(ep_dir / "video.mp4"))
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"{episode_id}: video={num_frames}フレーム, actions.csv={len(actions_df)}フレーム")

n_samples = min(8, num_frames)
sample_idx = np.linspace(0, num_frames - 1, num=n_samples, dtype=int)

sample_frames = []
for idx in sample_idx:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, frame_bgr = cap.read()
    sample_frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB) if ok else None)
cap.release()

fig, axes = plt.subplots(1, n_samples, figsize=(3 * n_samples, 3))
if n_samples == 1:
    axes = [axes]
for ax, idx, frame in zip(axes, sample_idx, sample_frames):
    action_idx = int(actions_df.loc[actions_df["frame_idx"] == idx, "action"].iloc[0])
    if frame is not None:
        ax.imshow(frame)
    ax.set_title(ACTION_NAMES[action_idx], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## アクション系列の時系列プロット

選択したエピソード内で、どのタイミングでどのアクションを取ったかを確認できる(ジャンプのタイミング等)。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(actions_df["frame_idx"], actions_df["action"])
ax.set_xlabel("フレーム番号")
ax.set_ylabel("アクション")
ax.set_yticks(range(len(SIMPLE_MOVEMENT)))
ax.set_yticklabels(ACTION_NAMES)
ax.set_title(f"{episode_id} のアクション系列")
plt.tight_layout()
plt.show()

## データのクリーニング

数フレームしかない、記録ミスなど明らかにおかしいエピソードが見つかった場合は、
該当する `data/episodes/ep_XXXX/` フォルダ(`video.mp4` と `actions.csv`)を削除し、
`data/manifest.csv` から対応する行を手動で削除してください。